In [ ]:
# data-agent notebook bootstrap
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

try:
    from IPython.display import display
except Exception:
    def display(value):
        print(value)

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
try:
    from IPython.display import display
except ImportError:
    display = print

%matplotlib inline

In [ ]:
# Load the unified dataset
data_path = Path("unified_dataset.jsonl")
df = pd.read_json(data_path, lines=True)
print(f"Loaded {len(df)} rows")

In [ ]:
# Preview and schema
print("Schema:")
print(df.dtypes)
print("\nFirst 3 rows:")
display(df.head(3))

In [ ]:
# Missingness analysis
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
missing_df = pd.DataFrame({"missing": missing, "pct": missing_pct})
print("Missing values:")
display(missing_df[missing_df["missing"] > 0])

In [ ]:
# Label distribution (only 50 rows have labels)
if "label" in df.columns and df["label"].notna().any():
    label_counts = df["label"].value_counts()
    print(f"Labeled rows: {df['label'].notna().sum()}/{len(df)}")
    print(f"Unique labels: {df['label'].nunique()}")
    plt.figure(figsize=(8, 4))
    sns.barplot(x=label_counts.values, y=range(len(label_counts)), orient="h")
    plt.title("Label Distribution")
    plt.xlabel("Count")
    plt.tight_layout()
    plt.show()
else:
    print("No labels present")

In [ ]:
# Text length distribution
if "text" in df.columns:
    df["text_len"] = df["text"].str.split().str.len()
    print(f"Text length stats (words): mean={df['text_len'].mean():.1f}, median={df['text_len'].median():.1f}")
    plt.figure(figsize=(8, 4))
    sns.histplot(df["text_len"], bins=30, kde=True)
    plt.title("Text Length Distribution")
    plt.xlabel("Word Count")
    plt.tight_layout()
    plt.show()
else:
    print("No text column")

In [ ]:
# Source distribution
if "source" in df.columns:
    source_counts = df["source"].value_counts()
    print("Source distribution:")
    display(source_counts)
    plt.figure(figsize=(8, 4))
    sns.barplot(x=source_counts.values, y=source_counts.index, orient="h")
    plt.title("Source Distribution")
    plt.xlabel("Count")
    plt.tight_layout()
    plt.show()

## Data Quality Review

### Analyzer View

- Task interpretation: unknown
- Primary modality: text
- Target semantics: unknown
- Relevant checks: text_length_distribution, label_format_validity, source_coverage, answer_format_consistency
- Lower-value checks: class_balance_for_label, numeric_outlier_detection, audio_modality_checks
- Priority actions: validate_label_format_has_solution_marker, check_russian_text_quality, verify_project_euler_problem_format

### Strategy Justification

This is a math reasoning text2text dataset. The 'missing' values (70 null labels) are intentional - they are unlabeled math problems from all-russian (60) and project-euler (10) sources. Median imputation is inappropriate for text solutions. No numeric outliers exist in this text dataset. Deduplication on normalized text removed 0 true duplicates. All 120 rows preserved: 50 labeled for supervised training, 70 unlabeled preserved for future label generation.

- Missing values: `not_applicable`
- Duplicates: `drop_normalized_text`
- Outliers: `not_applicable`

### Findings

- Missing values before cleaning: 300
- Duplicate rows before cleaning: 0
- Numeric outliers before cleaning: 0
- Imbalance column: `label`
- Majority class share after cleaning: not applicable

### Before / After

- Missing values: 0 -> 0
- Duplicates: 0 -> 0
- Outliers: 0 -> 0


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

raw_df = pd.read_json('data/collection/unified_dataset.jsonl', lines=True)
clean_df = pd.read_json('data/quality/cleaned_dataset.jsonl', lines=True)
primary_modality = 'text'

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
missing_counts = raw_df.isna().sum().sort_values(ascending=False)
missing_counts = missing_counts[missing_counts > 0]
if not missing_counts.empty:
    sns.barplot(x=missing_counts.values, y=missing_counts.index, ax=axes[0, 0], color='#d97706')
    axes[0, 0].set_title('Missing values by column')
else:
    axes[0, 0].text(0.5, 0.5, 'No missing values', ha='center', va='center')
    axes[0, 0].set_axis_off()

if {'source', 'label'}.issubset(raw_df.columns):
    coverage = raw_df.assign(label_present=raw_df['label'].notna()).groupby('source', dropna=False)['label_present'].mean().sort_values(ascending=False).head(10)
    if not coverage.empty:
        sns.barplot(x=coverage.values, y=coverage.index.astype(str), ax=axes[0, 1], color='#2563eb')
        axes[0, 1].set_title('Label coverage by source')
        axes[0, 1].set_xlim(0, 1)
    else:
        axes[0, 1].text(0.5, 0.5, 'No source coverage data', ha='center', va='center')
        axes[0, 1].set_axis_off()
elif 'source' in raw_df.columns:
    source_counts = raw_df['source'].astype(str).value_counts().head(10)
    sns.barplot(x=source_counts.values, y=source_counts.index, ax=axes[0, 1], color='#2563eb')
    axes[0, 1].set_title('Top sources')
else:
    axes[0, 1].text(0.5, 0.5, 'No source column', ha='center', va='center')
    axes[0, 1].set_axis_off()

numeric_columns = raw_df.select_dtypes(include=['number']).columns.tolist()
if numeric_columns and not (len(numeric_columns) == 1 and 'label' in numeric_columns and primary_modality == 'text'):
    sns.boxplot(data=raw_df[numeric_columns], orient='h', ax=axes[1, 0], color='#f59e0b')
    axes[1, 0].set_title('Raw numeric distributions')
    sns.boxplot(data=clean_df[numeric_columns], orient='h', ax=axes[1, 1], color='#10b981')
    axes[1, 1].set_title('Cleaned numeric distributions')
else:
    if 'text' in raw_df.columns:
        raw_lengths = raw_df['text'].fillna('').astype(str).str.split().str.len()
        clean_lengths = clean_df['text'].fillna('').astype(str).str.split().str.len()
        sns.histplot(raw_lengths, bins=30, ax=axes[1, 0], color='#f59e0b')
        axes[1, 0].set_title('Raw text length distribution')
        sns.histplot(clean_lengths, bins=30, ax=axes[1, 1], color='#10b981')
        axes[1, 1].set_title('Cleaned text length distribution')
    else:
        axes[1, 0].text(0.5, 0.5, 'No numeric columns', ha='center', va='center')
        axes[1, 0].set_axis_off()
        axes[1, 1].text(0.5, 0.5, 'No numeric columns', ha='center', va='center')
        axes[1, 1].set_axis_off()

plt.tight_layout()
plt.show()
